In [ ]:
# Import necessary libraries
import pandas as pd
import requests
import numpy as np
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup as bs
import json
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
from PIL import Image
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from googleapiclient.discovery import build
import nltk
from nltk import tokenize
import pycountry
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from kneed import KneeLocator
from matplotlib import pyplot as plt
import plotly.graph_objects as go
from collections import Counter
import warnings
from collections import Counter
warnings.filterwarnings('ignore')


In [ ]:
### Web-Scraping Data from Chartmaster for Debut Album Streams

# Functions to extract data from ChartMasters
def extract_album_chart(table_object, album_name):
    '''This function returns a dataframe containing popularity information for each album. Table_object refers 
    to the beautiful_soup object containing the data. Album_name is the string of the album name'''
    # get list of items from soup object
    header_table = [th.get_text().strip() for th in table_object.find_all('th')] # extract table headers
    rows_table = table_object.find_all('tr')[1:] # extract row data
    data_table = [[td.get_text().strip() for td in row.find_all('td')] for row in rows_table]
    album_data = [row.find_all('td')[5].get_text().strip() for row in rows_table] # album data

    # convert list items to a dataframe
    df = pd.DataFrame(data_table, columns=header_table).assign(Album=album_name)
    df['Debut Sales'] = album_data # add column
    df = df.drop(['Average', 'Songs', 'Top'], axis=1) # drop columns
    return(df)

def get_country_code(country):
    '''This function returns the ISO alpha-3 code for a country name'''
    if country == 'United States of America':
        return 'USA'
    elif country == 'Global':
        return 'OWID_WRL'
    elif country == 'South Korea':
        return 'KOR'
    elif country == 'Czech Republic':
        return 'CZE'
    elif country == 'Taiwan':
        return 'TWN'
    else:
        try:
            return pycountry.countries.get(name=country).alpha_3
        except AttributeError:
            return None
        
# Web-scrape global chart data for Drake albums
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2) AppleWebKit/601.3.9 (KHTML, like Gecko) Version/9.0.2 Safari/601.3.9'} # create header

url_song_data = 'https://chartmasters.org/spotify-top-album-debuts/?view=artist&artist_name=drake&artist_id=' # url for website with data
response = requests.get(url_song_data, headers=headers) # get response
bs_object = bs(response.content, 'html.parser') # create beautiful soup object 
table_views = bs_object.find_all('table')[0] # get 'views' data 
table_more_life = bs_object.find_all('table')[1] # 'get more life' data 
table_scorpion = bs_object.find_all('table')[2] # 'get scorpion' data
table_dldt = bs_object.find_all('table')[3] # get 'dark lane demo tapes' data
table_clb = bs_object.find_all('table')[4] # get 'clb' data 
table_hn = bs_object.find_all('table')[5] # get 'honestly, nevermind' data
table_hl = bs_object.find_all('table')[6] # get 'her loss' data

# create dataframe
views_df = extract_album_chart(table_views, "Views")
morelife_df = extract_album_chart(table_more_life, "More Life")
scorpion_df = extract_album_chart(table_scorpion, "Scorpion")
dldt_df = extract_album_chart(table_dldt, "Dark Lane Demo Tapes")
clb_df = extract_album_chart(table_clb, "Certified Lover Boy")
hn_df = extract_album_chart(table_hn, "Honestly, Nevermind")
hl_df = extract_album_chart(table_hl, "Her Loss")

# Data pre-processing
discography = pd.concat([views_df, morelife_df, scorpion_df, dldt_df, clb_df, hn_df, hl_df], ignore_index=True) # Concatenate the dataframes
duplicates = discography[discography.duplicated()] # Identify duplicates
discography = discography.drop_duplicates() # Drop duplicates

# Merge the concatenated dataframe on the 'Country', 'Streams', 'Album', and 'Views Debut' columns
merged = pd.merge(discography, discography, on=['Country', 'Streams', 'Album', 'Debut Sales'], how='right')
del merged["_y"] # remove columns
del merged["_x"]

# Top streams from US and Global from each Album 
subset = merged[['Country', 'Album', 'Streams']] # Subset the merged dataset to include only the relevant columns
# Filter the dataset to include only rows where the 'Country' column is either 'USA' or 'Global'
filtered = subset[subset['Country'].isin(['United States of America', 'Global'])] 
# Group the dataset by 'Country' and 'Album' and sort each group by 'Streams' in descending order
grouped = filtered.groupby(['Country', 'Album']).apply(lambda x: x.sort_values('Streams', ascending=False)) 
top5 = grouped.groupby(['Country', 'Album']).head(5) # Select the top 5 rows of each group using the head() method

# Concatenate the results for the USA and Global groups
usa_top5 = top5[top5['Country'] == 'United States of America']
global_top5 = top5[top5['Country'] == 'Global']
tops = pd.concat([usa_top5, global_top5], ignore_index=True)
print(tops)


In [ ]:
### Choropleth Map for Drake Debut Album Streams
merged['Streams'] = merged['Streams'].astype(str) # Convert the column to string type
merged['Streams'] = merged['Streams'].str.replace(',', '').astype(int) # .str to the columns
tot = merged.groupby('Country')['Streams'].sum().reset_index() # finding the total number of streams from each album in each country
total_streams = tot.reset_index() # convert back to dataframe
total_streams['Country Code'] = total_streams['Country'].apply(get_country_code)

#Create a choropleth map using the 'choropleth' function 
fig = px.choropleth(total_streams, locations='Country Code', 
                        color='Streams',
                        color_continuous_scale='twilight',
                        range_color=[0,1500000000],
                        color_continuous_midpoint=100000000,
                        hover_name='Country',
                        title='Drake Total Debut Stream Popularity',
                        projection='natural earth')
fig.update_layout(coloraxis_colorbar=dict(
    tickmode='linear', # Set the tick mode to linear
    dtick=200000000, 
    tick0=0 
))
fig.show()


In [ ]:
### Box Plot for Drake Debut Album Streams
# add numbers to boxplot 
tops['Streams'] = tops['Streams'].astype(str) # Convert the column to string type
tops['Streams'] = tops['Streams'].str.replace(',', '').astype(int) # apply the .str accessor on the column

# Group and plot the data
grouped_data = tops.groupby(['Country', 'Album'])['Streams'].sum().unstack()
ax = grouped_data.plot(kind='bar', stacked=True, figsize=(10, 6))
ax.set_title('Top Drake Album Streams Global vs United States')
ax.set_xlabel('Country')
ax.set_ylabel('Total Streams (in 100 millions)')


In [ ]:
### Web-Scrape Data for Drake's Top 40 Songs
url = 'https://www.officialcharts.com/chart-news/drakes-official-top-40-most-streamed-songs__21625/'
page = requests.get(url)
soup = bs(page.text, "html.parser") # convert string into BeautifulSoup object
table = soup.find_all('table') # using find_all method get all tables

# convert soup item to dataframe
res1 = [] # initialize empty list
for tr in table[0].find_all('tr'): # iterate over for loop 
    td = tr.find_all('td') # find_all method for 'td' 
    row = [tr.text.strip() for tr in td if tr.text.strip()] # strip our data
    if row:
        res1.append(row) # append data in our empty list

ranked_songs = pd.DataFrame(res1, columns=["Position", "Title", "Artist","Peak", "Year"]) # create pandas dataframe
ranked_songs = ranked_songs.drop(0) # drop row 0
ranked_songs['Title'] = ranked_songs['Title'].apply(str.lower) # lowercase song titles

ranked_songs.loc[25, 'Year'] = 2018 # fixing error from website

 # changing specific song names to match with spotify API data
ranked_songs.loc[16, 'Title'] = 'hold on, we\'re going home'
ranked_songs.loc[24, 'Title'] = 'don’t matter to me' 
ranked_songs.loc[29, 'Title'] = 'wanna know remix'
ranked_songs.loc[32, 'Title'] = 'what\'s my name?' 
ranked_songs.loc[36, 'Title'] = 'marvins room' 
ranked_songs['Year'] = ranked_songs['Year'].astype(int) # convert year column to int
ranked_songs = ranked_songs.sort_values(by=['Year'])

### Box-plot
fig, ax = plt.subplots(figsize=(20, 10)) # change plot size
p2 = sns.countplot(x=ranked_songs["Year"]) # create box plot 
for p in p2.patches: # get counts for each year
    p2.annotate('{:.1f}'.format(p.get_height()), (p.get_x()+0.25, p.get_height()+0.01))
plt.xlabel("Year" , size = 12) # set axis
plt.ylabel("Frequency" , size = 12)
plt.title("Frequency of Top Songs by Year" , size = 24)
plt.show()


In [ ]:
### Extract data from Spotify API
# Create API keys
URL = 'https://api.spotify.com/v1/' # base url for spotify
MY_ID = '47dd696c104e49d1b6a9925ad26a829d' # client access ID
MY_SECRET = '0e52b5a013a64b058001e2f69d22ab96' # client secret ID
AUTH_URL = 'https://accounts.spotify.com/api/token' # authentication url
auth_response = requests.post(AUTH_URL, { # create authentication response
    'grant_type': 'client_credentials',
    'client_id': MY_ID,
    'client_secret': MY_SECRET,
})
my_key = auth_response.json()['access_token'] # get access token
headers = {
    'Authorization': 'Bearer {token}'.format(token=my_key) # create header for our API
}

# Functions to extract data from playlist
def extract_tracks(Playlist_ID):
    '''This function returns a list of tuples containing track name, id, popularity, album release date, album name. 
    Input requires the playlist ID from spotify.'''
    
    # Get first 50 results; Spotify API only returns the first 50 results at once
    response_albums = requests.get(URL + 'playlists/' + Playlist_ID + '/tracks', # url to get items for playlist
                               headers=headers, 
                               params={'additional_types': 'track', 'limit': 50,'market': 'US','offset': 0}) # parameters
    d1 = response_albums.json() # convert response into JSON list 
    songs_info = list() # initialize empty list
    for song in d1['items']:   # add track information to the list
        songs_info.append((song['track']['name'], song['track']['id'], song['track']['popularity'], # extract information from json
                           song['track']['album']['release_date'], song['track']['album']['name'])) 
    i = 0  # get data from next 50 entries
    while (i < d1['total']): # while loop to repeat code
        next_link = d1['next'] # extract URL for next 50 songs to extract
        if next_link is None: # conditional: if url is empty, end loop
            break   
        response_albums = requests.get(next_link, headers=headers) # extract data from URL
        d1 = response_albums.json() # convert response into JSON
        for song in d1['items']: # append song info to our list
            songs_info.append((song['track']['name'], song['track']['id'], song['track']['popularity'], song['track']['album']['release_date'], song['track']['album']['name']))
        i += 50 # add 50 to i and re-run loop
    return(songs_info) # return list

def extract_features(Playlist_ID):
    '''This function returns list of attributes for each song based on an input Playlist ID from Spotify.'''
    input_tuple = extract_tracks(Playlist_ID) # get track information from extract_tracks function
    data = list() # initialize empty list
    for i in range(0,len(input_tuple)): # run loop through song information list
        response_track = requests.get(URL + 'audio-features/' + input_tuple[i][1], headers=headers) # add song ID from tuple to url
        response_track = response_track.json() # convert response into json
        response_track.update({
            'track_popularity': input_tuple[i][2], # add track popularity from song info list
            'track_name': input_tuple[i][0], # add track name
            'release_date': input_tuple[i][3], # add album release date
            'album_name': input_tuple[i][4] # add album name 
        })
        data.append(response_track) # append info into a list  
    return(data)

playlist_id = '7b46c5syjtG86a77R7SnMs' # Drake playlist containing all the songs
data_list = extract_features(playlist_id) # get list of song attributes based on Drake's playlist ID

df = pd.DataFrame(data_list) # convert to Pandas Dataframe

x = list() # empty list
for i in range(0,len(df)): # add year column 
    x.append(df['release_date'][i][0:4])
    
df['Year'] = x # add 'Year' column   
df['release_date'] = pd.to_datetime(df['release_date']) # change release_date to datetime
df = df.sort_values(by='release_date') # sort by release date; oldest to newest
df = df.drop(['type', 'id', 'uri','track_href','analysis_url'],axis = 1) # remove unnecessary columns
df['track_name'] = df['track_name'].apply(str.lower) # lowercase all track names
df['album_name'] = df['album_name'].apply(str.lower) # lowercase all album names


In [ ]:
### Create Dataframe of just song attributes
song_att = df.drop(['track_name','release_date','album_name','Year'], axis = 1)
# Distribution of track popularity
fig, ax = plt.subplots(figsize=(20, 10))
p1 = sns.histplot(data=song_att, x= 'track_popularity')
for p in p1.patches:
    ax.annotate(f'{p.get_height():.0f}\n',
                (p.get_x() + p.get_width() / 2, p.get_height()), ha='center', va='center', color='black')
plt.xlabel( "Track Popularity" , size = 12 )
plt.ylabel( "Frequency" , size = 12 )
plt.title( "Distribution of Track Popularity" , size = 24 )
plt.show()


In [ ]:
### Correlation Heat-Map between all song attributes
fig, ax = plt.subplots(figsize=(20, 10)) # set figure size
sns.heatmap(song_att.corr(), annot=True, linewidth=.5, cmap="Blues",ax = ax) # create heat plot
plt.title("Correlation Heat Plot of Song Attributes" , size = 24) 
plt.show()


In [ ]:
# Elbow Method
song_attribute_df = df.drop(['track_name','release_date','Year','track_popularity', 'album_name'], axis=1) # dataframe with only song attributes, no track popularity

# standardize our data for k-means
scaler = StandardScaler() 
X_std = scaler.fit_transform(song_attribute_df) # create data-frame
X_std = pd.DataFrame(X_std) # create dataframe

# Find number of clusters using Elbow Method
sse = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', max_iter=300, n_init=10, random_state=0)
    kmeans.fit(X_std)
    sse.append(kmeans.inertia_)
plt.plot(range(1, 11), sse)
plt.title('Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('Sum of Squares')
plt.show()


In [ ]:
k2 = KneeLocator(
    range(1, 11), sse, curve="convex", direction="decreasing"
)
k2.elbow


In [ ]:
# K-means clustering using scikit learn package
kmeans = KMeans(n_clusters=4, init='k-means++', random_state=0) 
kmeans.fit(X_std)
df['Cluster'] = kmeans.labels_ # add labels
song_attribute_df['Cluster'] = kmeans.labels_ # add labels

cluster_df = song_attribute_df.groupby("Cluster").mean() # get avg song attributes by cluter
cluster_df


In [ ]:
### Create Histogram of Top 40 Songs by Cluster
def strip_footnote(x):
    """This function removes bracketed footnotes, such as '[1]'."""
    if pd.isna(x):
        return x
    return x.partition("(")[0]

txt = df['track_name'].apply(strip_footnote)
df['track_name'] = txt
df['track_name'] = df['track_name'].str.strip()

txt2 = ranked_songs['Title'].apply(strip_footnote)
ranked_songs['Title'] = txt2
ranked_songs['Title'] = ranked_songs['Title'].str.strip()

condensed_df = df[['track_name', 'Cluster', 'track_popularity']]
top_songs =  condensed_df.set_index('track_name').join(ranked_songs.set_index('Title')) 
top_songs = top_songs.dropna()

fig, ax = plt.subplots(figsize=(20, 10))
p2 = sns.countplot(x=top_songs["Cluster"]) # frequency of 1's
for p in p2.patches:
    ax.annotate(f'{p.get_height():.0f}\n',
                (p.get_x() + p.get_width() / 2, p.get_height()), ha='center', va='center', color='black')
plt.xlabel( "Cluster" , size = 12 )
plt.ylabel( "Frequency" , size = 12 )
plt.title( "Bar Plot of Cluster vs Top 40 Songs" , size = 24 )
plt.show()


In [ ]:
### get comments from reddit post
url1 = 'https://www.reddit.com/r/Drizzy/comments/u2xpzp/best_drake_album/.json'
url2 = 'https://www.reddit.com/r/Drizzy/comments/q2t6f0/ranking_drakes_discography/.json'
url3 = 'https://www.reddit.com/r/Drizzy/comments/92rvjc/what_are_your_top_5_songs_on_scorpion/.json'
params = {'limit': 100,}
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2) AppleWebKit/601.3.9 (KHTML, like Gecko) Version/9.0.2 Safari/601.3.9'}

def get_reddit_comments(url, params, headers):
    response = requests.get(url, params=params, headers=headers) # get response
    data = response.json()
    comments = [] # Get the comments from the first set of comments
    for comment in data[1]['data']['children']:
        comments.append(comment['data']['body'])
    after = data[1]['data']['after'] # Get the 'after' parameter from the response, which is used to get the next set of comments
    
    while after is not None: # Loop through the next sets of comments until there are no more comments to retrieve
        params['after'] = after  # Add the 'after' parameter to the parameters and send a request to the Reddit API to get the next set of comments
        response = requests.get(url2, params=params, headers=headers)
        data = response.json()  # Extract the JSON data from the response
        for comment in data[1]['data']['children']: # Get the comments from the current set of comments
            comments.append(comment['data']['body'])
        after = data[1]['data']['after'] # Get the 'after' parameter from the response, which is used to get the next set of comments
    return(comments)

c1 = get_reddit_comments(url1, params, headers) # get comments from subreddit 1
c2= get_reddit_comments(url2, params, headers) # get comments from subreddit 2
c1 = [w.lower() for w in c1] # lowercase comments
c2 =[w.lower() for w in c2]


In [ ]:
c1[10] # example of comment


In [ ]:
c2[12] # example of comment


In [ ]:
keywords = ['nwts','take care', 'iyrtitl', 'views', 'clb', 'more life', 'thank me later', 'scorpion', 'so far gone','dldt'] # key words
album_keywords = [] # album keyword list

c = c1 + c2 # join lists
for comment in c:
    for keyword in keywords:
        if keyword in comment.lower():
            album_keywords.append(keyword)

keyword_counts = Counter(album_keywords)

keyword_counts_sorted = dict(sorted(keyword_counts.items(), key=lambda item: item[1], reverse=True))

# Create a bar plot of the frequency distribution with different colors for each keyword
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=list(keyword_counts_sorted.keys()), y=list(keyword_counts_sorted.values()), palette='muted', ax=ax)
ax.set_title('Frequency distribution of Drake album keywords')
ax.set_xlabel('Album keywords')
ax.set_ylabel('Frequency')
plt.xticks(rotation=45, ha='right')
plt.show()


In [ ]:
# Word Cloud
keyword_counts = Counter(album_keywords)
wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(keyword_counts)

# Display the word cloud
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Album Keywords')
plt.show()


In [ ]:
c3 = get_reddit_comments(url3, params, headers) # get comments from subreddit 1

keywords = ['survival', 'nonstop', 'elevate', 'emotionless', "god's plan", 'gods plan', "i'm upset", 'im upset', '8 out of 10', 'mob ties', 'cant take a joke', "can't take a joke", "sandra's rose", 'talk up', 'is there more' ]
album_keywords = []

for comment in c3:
    for keyword in keywords:
        if keyword in comment.lower():
            album_keywords.append(keyword)

            
keyword_counts = Counter(album_keywords)
keyword_counts_sorted = dict(sorted(keyword_counts.items(), key=lambda item: item[1], reverse=True))

# Create a bar plot of the frequency distribution with different colors for each keyword
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=list(keyword_counts_sorted.keys()), y=list(keyword_counts_sorted.values()), palette='muted', ax=ax)
ax.set_title('Frequency distribution of Top 5 Songs on Scorpion')
ax.set_xlabel('Song Keywords')
ax.set_ylabel('Frequency')
plt.xticks(rotation=45, ha='right')
plt.show()


In [ ]:
# Word Cloud
keyword_counts = Counter(album_keywords)
wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(keyword_counts)

# Display the word cloud
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Song Keywords')
plt.show()


In [ ]:
#from nltk.sentiment.vader import SentimentIntensityAnalyzer
api_key = "YOUR-API-KEY"
video_id = "-HAuK8Z6q6I"  #extracting from Drake's Accecptance Speech

#build a resource for youtube 
resource = build('youtube', 'v3', developerKey=api_key)

#create a request to get 100 comments on the video
request = resource. commentThreads().list(
                            part="snippet",
                            videoId=video_id,
                            maxResults= 100,
                            order="orderUnspecified") #top comments

response = request.execute() #execute the request

items = response['items'][:100]
df=pd.json_normalize(items)
#analyzing sentiment analysis on raw data to get a more true score
analyzer = SentimentIntensityAnalyzer()
df['vid1_compound'] = [analyzer.polarity_scores(x)['compound'] for x in df['snippet.topLevelComment.snippet.textDisplay']]
vid1compound = df['vid1_compound']
avg_vid1_comp=(df['vid1_compound'].mean()) 


df=df['snippet.topLevelComment.snippet.textDisplay'] #extracting the comments
df.head()
text = df.to_csv() #converting the data frame to text file, in order to generate to a word cloud of a shape in python
words = tokenize.word_tokenize(text)
words = [w for w in words if w.isalpha()] #assures only words are in the output 
stopwords = ["the", "a", "and", "or", "in", 'https', 'trump', 'is', 'you', 'of' 's', 'I', 'to', 't', 'br'] #assures only words with usefull meaning are in wordcloud
final_words = [w for w in words if w not in stopwords]
fq = nltk.FreqDist (w for w in final_words if w.isalnum())
text = ' '.join(fq) #turning list back into string

# Create a word cloud image
cloud = np.array(Image.open("cloud.png")) #the mask for our cloud
wc = WordCloud(stopwords = STOPWORDS, background_color="black", max_words=100, mask=cloud,
               contour_width=3, contour_color='firebrick')

# Generate a wordcloud
wc.generate(text)

# show and adjusting sizes for visualization purposes
plt.figure(figsize=[20,10])
plt.imshow(wc, interpolation='bilinear')
plt.title('Drake Artist of The Decade (Acceptance Speech)', fontsize=30)
plt.axis("off")
plt.show()


In [ ]:
video_id = "-CXxiFSmBUg"  #extracting from Drake The Biggest Fraud in Hip Hop
#build a resource for youtube
resource = build('youtube', 'v3', developerKey=api_key)

#create a request to get 100 comments on the video
request = resource. commentThreads().list(
                            part="snippet",
                            videoId=video_id,
                            maxResults= 100,
                            order="orderUnspecified") #top comments

response = request.execute() #execute the request

items = response['items'][:100]
df=pd.json_normalize(items)
#analyzing sentiment analysis on raw data to get a more true score
analyzer = SentimentIntensityAnalyzer()
df['vid2_compound'] = [analyzer.polarity_scores(x)['compound'] for x in df['snippet.topLevelComment.snippet.textDisplay']]
avg_vid2_comp=(df['vid2_compound'].mean()) 


df=df['snippet.topLevelComment.snippet.textDisplay'] #extracting the comments
df.head()
text = df.to_csv() #converting the data frame to text file, in order to generate to a word cloud of a shape in python
words = tokenize.word_tokenize(text)
words = [w for w in words if w.isalpha()] #assures only words are in the output 
stopwords = ["the", "a", "and", "or", "in", 'https', 'trump', 'is', 'you', 'of' 's', 'I', 'to', 't', 'br'] #assures only words with usefull meaning are in wordcloud
final_words = [w for w in words if w not in stopwords]
fq = nltk.FreqDist (w for w in final_words if w.isalnum())
text = ' '.join(fq) #turning list back into string

# Create a word cloud image
cloud = np.array(Image.open("cloud.png")) #the mask for our cloud
wc = WordCloud(background_color="black", max_words=100, mask=cloud,
               contour_width=3, contour_color='firebrick')

# Generate a wordcloud
wc.generate(text)

# show and adjusting sizes for visualization purposes
plt.figure(figsize=[20,10])
plt.imshow(wc, interpolation='bilinear')
plt.title('Drake: The Biggest Fraud in Hip Hop', fontsize=30)
plt.axis("off")
plt.show()


In [ ]:
print(avg_vid1_comp)
print(avg_vid2_comp)


In [ ]:
# Creating a plot to compare the compound score
left = [1, 2]
height = [20, 13]
tick_label = ['Drake: Acceptance Speech (vid1)', 'Drake: Artist Fraud (vid2)']

plt.bar(left, height, tick_label = tick_label, width = .50, color = ['blue', 'green'])
plt.xlabel('Videos')
plt.ylabel('Avg. Compound Score Percentage')
plt.legend()
plt.title('Average Compound Score Comparison')
plt.show
